# AIQ Pipeline Walkthrough

**Flow:** Module runs → Shows findings (first 10) → User reviews/modifies → Next module

| Module | What it does | "Detected" means |
|--------|-------------|------------------|
| A10 | Raw chunking | Nothing (just chunks) |
| A11 | Domain intelligence | Context signals extracted |
| A12 | Content normalization | Structured elements found |
| A13 | Structure & headings | Sections with missing/generic headings |
| A14 | Smart chunking | Multi-topic sections split |
| A22 | Metadata enrichment | Date/version references found |
| A30 | Semantic clarity | Clarity issues found |
| A31 | Content governance | Governance patterns found |
| A32 | Consistency checking | Contradictions found |

---
## Cell 1: Setup & Configuration

In [3]:
import sys, os
if os.path.basename(os.getcwd()) == 'examples':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

from aiq.pipeline import AIQConfig

# === USER CONFIGURATION ===
INPUT_FILE = 'examples/neuroloft_kb.html'
if not os.path.exists(INPUT_FILE): INPUT_FILE = 'neuroloft_kb.html'

PII_MODE = 'smart'
CHUNK_MIN_WORDS = 50
CHUNK_MAX_WORDS = 200
TOPIC_SHIFT_THRESHOLD = 0.1
USE_LLM = False
LLM_PROVIDER = ''
LLM_API_KEY = ''
LLM_MODEL = ''
DOMAIN_TYPE = ''
COMPANY_NAME = ''
EXTRA_ACRONYMS = {}
EXTRA_ACTORS = {}
AUTO_RESOLVE_THRESHOLD = 0.7
MAX_ROWS = 10
CONTEXT_SENTENCES = 2  # How many sentences to show: 1 = just the issue, 2 = issue + 1 before, 3 = issue + 2 before, etc.

config = AIQConfig(
    pii_mode=PII_MODE, chunk_min_words=CHUNK_MIN_WORDS, chunk_max_words=CHUNK_MAX_WORDS,
    topic_shift_threshold=TOPIC_SHIFT_THRESHOLD, strip_html=True,
    llm_provider=LLM_PROVIDER if USE_LLM else '', llm_api_key=LLM_API_KEY if USE_LLM else '',
    llm_model=LLM_MODEL if USE_LLM else '', domain_type=DOMAIN_TYPE, company_name=COMPANY_NAME,
    extra_acronyms=EXTRA_ACRONYMS, extra_actors=EXTRA_ACTORS,
)

def show(items, fmt, label='items'):
    total = len(items)
    for item in items[:MAX_ROWS]:
        print(fmt(item))
    if total > MAX_ROWS:
        print(f'  ... and {total - MAX_ROWS} more {label}')

print(f'Config: PII={PII_MODE}, chunks={CHUNK_MIN_WORDS}-{CHUNK_MAX_WORDS}, LLM={USE_LLM}, max_rows={MAX_ROWS}')

Config: PII=smart, chunks=50-200, LLM=False, max_rows=10


## Cell 2: Load Document

In [6]:
with open(INPUT_FILE, encoding='utf-8') as f:
    html = f.read()
print(f'Loaded: {INPUT_FILE} ({len(html):,} chars, ~{len(html.split()):,} words)')

Loaded: neuroloft_kb.html (15,498 chars, ~1,670 words)


## Cell 3: A10 — Raw Chunking

In [9]:
from aiq.a10 import RawChunker, A10Config
a10_out = RawChunker(A10Config(min_words=CHUNK_MIN_WORDS, max_words=CHUNK_MAX_WORDS, strip_html=True)).run(html)
baseline_chunks = a10_out.chunks
print(f'Baseline chunks: {len(baseline_chunks)}')
show(baseline_chunks, lambda c: f'  {c.chunk_id}: {c.words}w | {c.content[:80]}...', 'chunks')

Baseline chunks: 11
  raw_1: 185w |  SPLIT  Neuroloft handles all billing inquiries for customers across North Ameri...
  raw_2: 186w | Log into your account. Go to Billing &gt; Transaction History. Click "Request Re...
  raw_3: 149w | International payments require additional documentation. Customers must provide ...
  raw_4: 183w | As shown in Figure 2, the refund workflow has two paths based on the refund amou...
  raw_5: 191w | Users Up to 5 Up to 25 Unlimited . Storage 10 GB 100 GB Unlimited . Support Emai...
  raw_6: 187w | ERR-001 Card declined Contact card issuer . ERR-002 Insufficient funds Add funds...
  raw_7: 191w | All new customer accounts receive a free trial period of 14 days starting from t...
  raw_8: 195w | These prices are locked in for customers who signed up before March 2022. Existi...
  raw_9: 197w | For privacy compliance, customer information in support cases should be handled ...
  raw_10: 185w | For internal issue tracking see https://jira.internal.neuroloft

## Cell 4: A11 — Domain Intelligence

In [12]:
from aiq.a11 import DomainInferrer, A11Config
a11_out = DomainInferrer(A11Config(mode='rule_then_llm' if USE_LLM else 'rule_only', llm_call=config.llm_call, source_title=os.path.splitext(os.path.basename(INPUT_FILE))[0])).run(baseline_chunks)
domain_ctx = a11_out.data['domain_context']
if DOMAIN_TYPE: domain_ctx.domain_type = DOMAIN_TYPE
if COMPANY_NAME: domain_ctx.company_name = COMPANY_NAME
if EXTRA_ACRONYMS: domain_ctx.acronyms.update(EXTRA_ACRONYMS)
if EXTRA_ACTORS: domain_ctx.actors.update(EXTRA_ACTORS)
print(f'Domain: {domain_ctx.domain_type} (confidence: {domain_ctx.confidence})')
print(f'Company: {domain_ctx.company_name}')
print(f'Actors ({len(domain_ctx.actors)}): {dict(list(domain_ctx.actors.items())[:MAX_ROWS])}')
print(f'Acronyms ({len(domain_ctx.acronyms)}): {dict(list(domain_ctx.acronyms.items())[:MAX_ROWS])}')
print(f'Products: {domain_ctx.product_names[:MAX_ROWS]}')

Domain: support (confidence: 0.94)
Company: Enterprise
Actors (4): {'support': 'support', 'billing': 'billing', 'finance': 'finance', 'agent': 'agent'}
Acronyms (9): {'CRM': 'Customer Relationship Management', 'SLA': 'Service Level Agreement', 'SWIFT': 'Society for Worldwide Interbank Financial Telecommunication', 'SSO': 'Single Sign-On', 'CSM': 'Customer Success Manager', 'PO': 'Purchase Order', 'CVV': 'Card Verification Value', 'DPO': 'Data Protection Officer', 'VPN': 'Virtual Private Network'}
Products: ['Enterprise', 'Professional', 'Starter']


### ✏️ Review: Domain Context

In [15]:
# domain_ctx.domain_type = 'medical'
# domain_ctx.company_name = 'Acme Corp'
# domain_ctx.acronyms['MRN'] = 'Medical Record Number'
print(f'Finalized: {domain_ctx.domain_type}, {len(domain_ctx.acronyms)} acronyms')

Finalized: support, 9 acronyms


## Cell 5: A12 — Content Normalization

In [18]:
from aiq.a12 import Normalizer, A12Config
a12_out = Normalizer(A12Config(mode='rule_then_llm' if USE_LLM else 'rule_only', llm_call=config.llm_call, domain_type=domain_ctx.domain_type)).run(html)
normalized_html = a12_out.data['normalized_html']
print(f'Elements found: {len(a12_out.findings)}')
show(a12_out.findings, lambda e: f'  {e.element_id} [{e.element_type}]: {e.source_ref}' + (f' ({len(e.row_texts)} rows)' if e.row_texts else ''), 'elements')

Elements found: 15
  table_1 [table]: Table: Payment Processing Times (5 rows)
  table_2 [table]: Table: Wire Transfer Details (5 rows)
  table_3 [table]: Table: Plan Comparison (5 rows)
  table_4 [table]: Table: Common Error Codes (5 rows)
  table_5 [table]: Table: Recent Support Cases (3 rows)
  table_6 [table]: Table: Escalation Contacts (3 rows)
  figure_1 [figure]: Figure 1: Payment Processing Flow
  figure_2 [figure]: Figure 2: Refund Workflow
  figure_3 [figure]: Figure 3: Monthly Revenue Chart
  figure_5 [figure]: Figure 5: Account Tier Comparison Chart - image not available
  ... and 5 more elements


## Cell 6: A13 — Structure & Headings

In [21]:
from aiq.a13 import Structurer, A13Config
a13_out = Structurer(A13Config(mode='rule_then_llm' if USE_LLM else 'rule_only', llm_call=config.llm_call)).run(normalized_html)
sections = a13_out.data['sections']
print(f'Sections: {len(sections)} (fixed {a13_out.resolved} headings)')
show(sections, lambda s: f'  {s.section_id}: "{s.heading}" ({s.words}w)' + (f' [{s.heading_source}]' if s.heading_source != 'original' else ''), 'sections')

Sections: 24 (fixed 1 headings)
  sec_0: "Billing inquiries" (177w) [generated_rule]
  sec_1: "Refund Process" (88w)
  sec_2: "Payment Methods" (21w)
  sec_3: "Payment Processing Times" (184w)
  sec_4: "Wire Transfer Details" (72w)
  sec_5: "Payment Flow Overview" (177w)
  sec_6: "Account Setup and Configuration" (110w)
  sec_7: "Plan Comparison" (110w)
  sec_8: "Troubleshooting Payment Issues" (123w)
  sec_9: "Common Error Codes" (66w)
  ... and 14 more sections


## Cell 7: A14 — Smart Chunking (pass 1)

In [24]:
from aiq.a14 import SmartChunker, A14Config
a14_out = SmartChunker(A14Config(min_words=CHUNK_MIN_WORDS, max_words=CHUNK_MAX_WORDS, topic_shift_threshold=TOPIC_SHIFT_THRESHOLD)).run(sections)
chunks = a14_out.chunks
print(f'Chunks: {len(chunks)} (from {len(sections)} sections)')
show(chunks, lambda c: f'  {c.chunk_id}: {c.words}w | "{c.heading}"', 'chunks')

Chunks: 27 (from 24 sections)
  c1: 55w | "Billing inquiries"
  c2: 70w | "Billing inquiries"
  c3: 52w | "Billing inquiries"
  c4: 88w | "Refund Process"
  c5: 74w | "Payment Methods"
  c6: 131w | "Payment Processing Times"
  c7: 72w | "Wire Transfer Details"
  c8: 52w | "Payment Flow Overview - 1"
  c9: 125w | "Payment Flow Overview"
  c10: 53w | "Account Setup and Configuration"
  ... and 17 more chunks


## Cell 8: A22 — Metadata Enrichment

In [27]:
from aiq.a22 import MetadataEnricher, A22Config
a22_out = MetadataEnricher(A22Config(flag_stale=True, stale_months=12)).run(chunks)
dated = [c for c in chunks if c.metadata.get('a22_dates')]
print(f'Date references: {a22_out.detected}, Chunks with dates: {len(dated)}')
show(dated, lambda c: f'  {c.chunk_id}: {c.metadata["a22_dates"]}', 'dated chunks')

Date references: 7, Chunks with dates: 7
  c6: [{'year': 2022, 'month': 1, 'raw': 'January 2022', 'type': 'month_year', 'age_months': 52}, {'year': 2022, 'month': 4, 'raw': 'Q2 2022', 'type': 'quarter', 'age_months': 49}]
  c20: [{'year': 2022, 'month': 1, 'raw': 'January 15, 2022', 'type': 'month_year', 'age_months': 52}, {'year': 2022, 'month': 3, 'raw': 'March 2022', 'type': 'month_year', 'age_months': 50}]
  c21: [{'year': 2026, 'month': 4, 'raw': 'April 1, 2026', 'type': 'month_year', 'age_months': 1}]
  c22: [{'year': 2023, 'month': 2, 'raw': 'February 2023', 'type': 'month_year', 'age_months': 39}]
  c24: [{'year': 2026, 'month': 3, 'raw': 'March 15, 2026', 'type': 'month_year', 'age_months': 2}]
  c25: [{'year': 2026, 'month': 3, 'raw': '2026-03-12', 'type': 'iso', 'age_months': 2}]
  c26: [{'year': 2026, 'month': 1, 'raw': 'Q1 2026', 'type': 'quarter', 'age_months': 4}]


## Cell 9: A30 — Semantic Clarity

In [30]:
import re as _re

from aiq.a30 import ClarityChecker, A30Config
a30_out = ClarityChecker(A30Config(pronoun_mode='rule_fix', acronym_mode='rule_fix', vague_entity_mode='rule_fix', reference_mode='rule_fix', llm_call=config.llm_call)).run(chunks, domain_context=domain_ctx)
fixed = [f for f in a30_out.findings if f.fixed]
unfixed = [f for f in a30_out.findings if not f.fixed]
chunk_map = {c.chunk_id: c for c in chunks}

def get_context(chunk_content, evidence, n_sentences):
    """Get n_sentences of context around the evidence sentence."""
    sents = _re.split(r'(?<=[.!?])\s+', chunk_content)
    # Find which sentence contains the evidence
    match_idx = -1
    for i, s in enumerate(sents):
        if evidence[:30].lower() in s.lower():
            match_idx = i
            break
    if match_idx == -1:
        return evidence  # fallback
    start = max(0, match_idx - (n_sentences - 1))
    end = match_idx + 1
    context_sents = sents[start:end]
    result = ' '.join(context_sents)
    if start > 0:
        result = '... ' + result
    if end < len(sents):
        result = result + ' ...'
    return result

print(f'Clarity: {a30_out.detected} detected, {a30_out.resolved} resolved, {a30_out.remaining} remaining')

print(f'Resolved ({len(fixed)}):')
show(fixed, lambda f: f'  [{f.chunk_id}] {f.issue_type}: {f.fix_detail}', 'fixes')

print(f'Unresolved ({len(unfixed)}):  (showing {CONTEXT_SENTENCES} sentence(s) of context)')
for i, f in enumerate(unfixed[:MAX_ROWS]):
    c = chunk_map.get(f.chunk_id)
    heading = c.heading if c else '?'
    context = get_context(c.content, f.evidence, CONTEXT_SENTENCES) if c else f.evidence
    print(f'  {i+1}. [{f.chunk_id}] {f.issue_type} | "{f.original_term}"')
    print(f'     Context: "{context}"')
    print(f'     Location: chunk {f.chunk_id}, heading "{heading}"')
    print()
if len(unfixed) > MAX_ROWS:
    print(f'  ... and {len(unfixed) - MAX_ROWS} more issues')

Clarity: 28 detected, 19 resolved, 9 remaining
Resolved (19):
  [c1] pronoun: Replaced "They" with "the billing team"
  [c2] pronoun: Replaced "they" with "the billing team"
  [c2] pronoun: Replaced "reach out to them" with "reach out to the billing team"
  [c2] vague_entity: Replaced "The platform" with "Enterprise"
  [c3] acronym: Expanded to: CRM (Customer Relationship Management)
  [c3] acronym: Expanded to: SLA (Service Level Agreement)
  [c7] pronoun: Replaced "They" with "the Enterprise support team"
  [c7] acronym: Expanded to: SWIFT (Society for Worldwide Interbank Financial Telecommunication)
  [c11] acronym: Expanded to: SSO (Single Sign-On)
  [c12] acronym: Expanded to: CSM (Customer Success Manager)
  ... and 9 more fixes
Unresolved (9):  (showing 2 sentence(s) of context)
  1. [c3] vague_entity | "The team"
     Context: "Make sure to have your account number ready when you call. The team also manages subscription renewals and cancellations. ..."
     Location: chunk c3, 

### ✏️ Review: Clarity Issues

In [33]:
# for c in chunks:
#     if c.chunk_id == 'c5':
#         c.content = c.content.replace('their account', "the customer's account")
print(f'Ready for governance: {len(chunks)} chunks')

Ready for governance: 27 chunks


## Cell 10: A31 — Content Governance

In [36]:
from aiq.a31 import Classifier, A31Config
from collections import Counter
content_before = {c.chunk_id: c.content for c in chunks}
a31_out = Classifier(A31Config(pii_mode=PII_MODE, llm_call=config.llm_call)).run(chunks, domain_context=domain_ctx)
cats = Counter(f.tag.value for f in a31_out.findings)
changed = [(c, content_before.get(c.chunk_id, '')) for c in chunks if c.content != content_before.get(c.chunk_id, '')]
print(f'Governance: {a31_out.detected} findings ({dict(cats)}), {a31_out.resolved} removals')
print(f'\nContent changes ({len(changed)}):')
show(changed, lambda x: f'  {x[0].chunk_id}: {"EMPTIED" if x[0].words == 0 else f"CLEANED (-{len(x[1].split()) - len(x[0].content.split())}w)"}', 'changes')
review_chunks = [c for c in chunks if c.tag.value not in ('content', '') and c.words > 0]
if review_chunks:
    print(f'\nReview tags ({len(review_chunks)}):')
    show(review_chunks, lambda c: f'  {c.chunk_id} [{c.tag.value}]: {c.tag_reason}', 'tags')

Governance: 49 findings ({'vague_claim': 2, 'internal_only': 5, 'escalation': 5, 'pii': 16, 'placeholder': 2, 'broken_reference': 4, 'editorial': 6, 'metadata_leak': 9}), 8 removals

Content changes (8):
  c4: EMPTIED
  c7: CLEANED (-7w)
  c11: CLEANED (-12w)
  c13: CLEANED (-12w)
  c23: CLEANED (-30w)
  c24: CLEANED (-49w)
  c25: CLEANED (-36w)
  c27: EMPTIED

Review tags (5):
  c1 [vague_claim]: Vague claim: "take appropriate action"
  c6 [escalation]: Escalation without contact details: "contact support"
  c12 [broken_reference]: Broken reference: "See our FAQ"
  c16 [broken_reference]: Broken reference: "[Figure 5: Account Tier Comparison Chart - image not availab"
  c17 [broken_reference]: Broken reference: "image not available"


### ✏️ Review: Governance Decisions

In [39]:
# Restore: for c in chunks: if c.chunk_id == 'c8': c.content = content_before['c8']; c.words = len(c.content.split())
active_chunks = [c for c in chunks if c.words >= 10]
print(f'Active: {len(active_chunks)} chunks (removed {len(chunks) - len(active_chunks)} empty)')

Active: 25 chunks (removed 2 empty)


## Cell 11: A32 — Consistency Checking

In [42]:
from aiq.a32 import ConsistencyChecker, A32Config
a32_out = ConsistencyChecker(A32Config(auto_resolve_threshold=AUTO_RESOLVE_THRESHOLD, use_llm_judge=USE_LLM, llm_call=config.llm_call)).run(active_chunks, domain_context=domain_ctx)
print(f'Contradictions: {a32_out.detected}, auto-resolved: {a32_out.resolved}, pairs: {a32_out.data["pairs_evaluated"]}')
show(a32_out.findings, lambda f: f'\n  {f.finding_id}: {f.conflict_type} (conf={f.confidence_score:.2f})\n    A [{f.chunk_a_id}]: {f.evidence_a[:70]}...\n    B [{f.chunk_b_id}]: {f.evidence_b[:70]}...\n    Winner: {f.suggested_winner or "none"} | Action: {f.action_taken}', 'contradictions')

Contradictions: 2, auto-resolved: 0, pairs: 105

  c10__c11_process: process (conf=0.00)
    A [c10]: 3 steps starting: Visit neuroloft...
    B [c11]: 3 steps starting: Go to Settings &gt; Billing...
    Winner: none | Action: flagged_for_review

  c20__c21_numeric: numeric (conf=0.40)
    A [c20]: Effective January 15, 2022 , the following pricing changes apply to al...
    B [c21]: As of April 1, 2026 , Neuroloft pricing has been updated: Starter plan...
    Winner: b | Action: flagged_for_review


### ✏️ Review: Contradictions

In [45]:
# f.user_decision = 'select_a' | 'select_b' | 'keep_both'
for f in a32_out.findings:
    if f.action_taken == 'flagged_for_review': pass
print('Contradictions reviewed.')

Contradictions reviewed.


## Cell 12: Re-chunk + Enrichment (pass 2)

In [48]:
import re
from aiq.a13.structurer import Section
from collections import Counter as _Counter
active_chunks = [c for c in active_chunks if c.words >= 10]
cleaned_sections = [Section(section_id=c.chunk_id, heading=c.heading, heading_level=2, content=c.content, words=c.words) for c in active_chunks]
final_chunks = SmartChunker(A14Config(min_words=CHUNK_MIN_WORDS, max_words=CHUNK_MAX_WORDS, topic_shift_threshold=TOPIC_SHIFT_THRESHOLD)).run(cleaned_sections).chunks
_stop = {'the','and','for','with','from','this','that','are','was','has','have','will','can','our','your','all','also','not','been','may','must'}
for c in final_chunks:
    if c.heading and not c.content.startswith(c.heading):
        kw = ', '.join(w for w, _ in _Counter(w for w in re.findall(r'\b[a-z]{4,}\b', c.content.lower()) if w not in _stop).most_common(5))
        c.content = f'{c.heading}. {kw}. {c.content}' if kw else f'{c.heading}. {c.content}'
        c.words = len(c.content.split())
print(f'Final chunks: {len(final_chunks)}')
show(final_chunks, lambda c: f'  {c.chunk_id}: {c.words}w | {c.content[:100]}...', 'chunks')

Final chunks: 24
  c1: 64w | Billing inquiries. billing, customers, team, neuroloft, handles. Neuroloft handles all billing inqui...
  c2: 80w | Billing inquiries. team, payment, business, billing, refunds. If there are any issues with a payment...
  c3: 65w | Billing inquiries. team, make, sure, account, number. Make sure to have your account number ready wh...
  c4: 81w | Payment Methods. processing, payment, times, time, fees. Neuroloft currently supports the following ...
  c5: 131w | Payment Processing Times — Check: Processing Time: 10-15 business days. Status: Deprecated - use wir...
  c6: 74w | Wire Transfer Details: field, value. Wire Transfer Details — Bank: Value: First National Bank. Wire ...
  c7: 62w | Payment Flow Overview - 1. diagram, through, payment, shown, above. The diagram below shows how paym...
  c8: 133w | Payment Flow Overview. refund, figure, workflow, amount, request. Refer to Figure 2 for the refund w...
  c9: 109w | Account Setup and Configuration. step, a

## Cell 13: Pipeline Summary

In [51]:
from collections import Counter

print('=' * 60)
print('AIQ PIPELINE SUMMARY')
print('=' * 60)

# Output stats
baseline_words = sum(c.words for c in baseline_chunks)
final_words = sum(c.words for c in final_chunks)
print(f'Input:  {len(baseline_chunks)} chunks, {baseline_words} words')
print(f'Output: {len(final_chunks)} chunks, {final_words} words')
print(f'Change: {len(final_chunks) - len(baseline_chunks):+d} chunks, {final_words - baseline_words:+d} words')
print(f'Time:   {"< 1s (pattern-based)" if not USE_LLM else "with LLM calls"}')

# Module summary table
print(f'{"─" * 60}')
print(f'  {"Module":<30s} {"Found":>6s} {"Resolved":>9s} {"Remaining":>10s}')
print(f'  {"─" * 55}')
modules = [
    ('Domain Intelligence', a11_out),
    ('Content Normalization', a12_out),
    ('Structure & Headings', a13_out),
    ('Metadata Enrichment', a22_out),
    ('Semantic Clarity', a30_out),
    ('Content Governance', a31_out),
    ('Consistency', a32_out),
]
total_det = total_res = total_rem = 0
for name, out in modules:
    # For governance, show actions (resolved) not raw pattern count in remaining
    remaining = out.remaining if name != 'Content Governance' else 0
    print(f'  {name:<30s} {out.detected:>6d} {out.resolved:>9d} {remaining:>10d}')
    total_det += out.detected
    total_res += out.resolved
    total_rem += remaining
print(f'  {"─" * 55}')
print(f'  {"TOTAL":<30s} {total_det:>6d} {total_res:>9d} {total_rem:>10d}')

print(f'{"=" * 60}')
print(f'Output: {len(final_chunks)} clean chunks ready for indexing')
print(f'{"=" * 60}')

AIQ PIPELINE SUMMARY
Input:  11 chunks, 1868 words
Output: 24 chunks, 2110 words
Change: +13 chunks, +242 words
Time:   < 1s (pattern-based)
────────────────────────────────────────────────────────────
  Module                          Found  Resolved  Remaining
  ───────────────────────────────────────────────────────
  Domain Intelligence               171       171          0
  Content Normalization              15        15          0
  Structure & Headings                1         1          0
  Metadata Enrichment                 7         7          0
  Semantic Clarity                   28        19          9
  Content Governance                 49         8          0
  Consistency                         2         0          2
  ───────────────────────────────────────────────────────
  TOTAL                             273       221         11
Output: 24 clean chunks ready for indexing


## Cell 14: Save Results

In [55]:
import json, os
from datetime import datetime

results_dir = os.path.join(os.path.dirname(os.getcwd()), 'results') if os.path.basename(os.getcwd()) == 'examples' else os.path.join(os.getcwd(), 'results')
os.makedirs(results_dir, exist_ok=True)

# Serialize findings
def serialize_findings(findings):
    out = []
    for f in findings:
        d = {}
        for attr in dir(f):
            if attr.startswith('_'): continue
            val = getattr(f, attr)
            if callable(val): continue
            if hasattr(val, 'value'): val = val.value  # enums
            if isinstance(val, (str, int, float, bool, list, dict, type(None))):
                d[attr] = val
        out.append(d)
    return out

results = {
    'timestamp': datetime.now().isoformat(),
    'input_file': INPUT_FILE,
    'config': {
        'pii_mode': PII_MODE,
        'chunk_min_words': CHUNK_MIN_WORDS,
        'chunk_max_words': CHUNK_MAX_WORDS,
        'use_llm': USE_LLM,
    },
    'stats': {
        'baseline_chunks': len(baseline_chunks),
        'baseline_words': sum(c.words for c in baseline_chunks),
        'final_chunks': len(final_chunks),
        'final_words': sum(c.words for c in final_chunks),
    },
    'modules': {},
    'baseline_chunks': [],
    'chunks': [],
    'findings': {
        'a30': serialize_findings(a30_out.findings),
        'a31': serialize_findings(a31_out.findings),
        'a32': serialize_findings(a32_out.findings),
    },
    'domain_context': {
        'domain_type': domain_ctx.domain_type,
        'company_name': domain_ctx.company_name,
        'actors': dict(domain_ctx.actors),
        'acronyms': dict(domain_ctx.acronyms),
        'products': domain_ctx.product_names,
    },
}

for name, out in modules:
    remaining = out.remaining if name != 'Content Governance' else 0
    results['modules'][name] = {'detected': out.detected, 'resolved': out.resolved, 'remaining': remaining}

for c in baseline_chunks:
    results['baseline_chunks'].append({'chunk_id': c.chunk_id, 'heading': c.heading, 'content': c.content, 'words': c.words})

for c in final_chunks:
    results['chunks'].append({'chunk_id': c.chunk_id, 'heading': c.heading, 'content': c.content, 'words': c.words})

doc_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
out_path = os.path.join(results_dir, f'{doc_name}_results.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f'Saved: {out_path}')
print(f'  Baseline: {len(results["baseline_chunks"])} chunks')
print(f'  Processed: {len(results["chunks"])} chunks')
print(f'  Findings: A30={len(results["findings"]["a30"])}, A31={len(results["findings"]["a31"])}, A32={len(results["findings"]["a32"])}')
print(f'  Domain: {results["domain_context"]["domain_type"]}, {len(results["domain_context"]["acronyms"])} acronyms')

Saved: C:\Users\samir\OneDrive\Desktop\KG RAG\opensource\aiq\results\neuroloft_kb_results.json
  Baseline: 11 chunks
  Processed: 24 chunks
  Findings: A30=28, A31=49, A32=2
  Domain: support, 9 acronyms
